# 실행 결과 기반 분석 보고서 생성

최종 분석 01~08의 검증된 결과와 보고서 근거 노트북의 표를 이용한다. 모형을 다시 적합하지 않는다. 한국어 해석·신뢰구간·표본 한계를 함께 담은 독립 HTML 보고서를 생성한다. 먼저 같은 폴더의 01_보고서_근거.ipynb를 실행한다.

보고서 대상: 졸업 프로젝트 연구자·지도교수. 기술 보고서 구성에서 지표 정의를 결과보다 앞에 두어 오독을 줄인다. 부정확한 인과 주장이나 유의성에 따른 모형 선택은 하지 않는다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
from datetime import datetime, timezone, timedelta
import json, hashlib, sqlite3, base64, html, shutil, subprocess
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = PROJECT_ROOT
R = project_path('reports/20261006')
OUT = R/'analysis_report'
E = OUT/'evidence'
OUT.mkdir(parents=True,exist_ok=True)
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def read_json(path): return project_json_loads(path.read_text(encoding='utf-8'))
execution = read_json(R/'execution_manifest.json')
assert execution['status']=='completed' and len(execution['steps'])==8
fe_meta = read_json(R/'09_analysis_summary.json')
gee_meta = read_json(R/'10_gee_analysis_summary.json')
for path,digest in fe_meta['output_sha256'].items(): assert sha(project_path(path))==digest
for name,digest in gee_meta['output_sha256'].items(): assert sha(R/name)==digest
assert sha(project_path(fe_meta['input_file']))==fe_meta['input_sha256']
print('최종 실행·모형 결과 해시 검증 완료')


## 1. 보고서에 필요한 데이터만 읽기

원래 회귀 결과와 구간을 유지한다. SQL은 저장된 결과 CSV를 조인·집계해 보고서 행을 생성한 실제 쿼리이며, 회귀 적합 자체는 기존 최종 노트북에 기록돼 있다.

In [ ]:
paths = {
 'industry': E/'01_industry_summary.csv', 'district':E/'02_district_summary.csv',
 'store_size':E/'03_store_size_coverage.csv','sales':E/'04_sales_contrasts.csv',
 'night':E/'05_night_contrasts.csv','effects':R/'09_industry_effects.csv',
 'omnibus':R/'09_omnibus_tests.csv','samples':R/'09_model_samples.csv',
 'gee':R/'10_gee_industry_effects.csv','comparison':R/'10_fe_gee_comparison.csv',
 'information':R/'09_information_concentration.csv','support':R/'09_activity_support.csv'}
connection = sqlite3.connect(':memory:')
frames={}
for name,path in paths.items():
    frames[name]=pd.read_csv(path)
    frames[name].to_sql(name,connection,index=False,if_exists='replace')
input_hashes={path.relative_to(ROOT).as_posix():sha(path) for path in paths.values()}
scope=read_json(E/'scope_summary.json')
assert len(frames['sales'])==len(frames['night'])==10
assert not frames['effects'].query("scenario=='main_500'")['p_fdr_primary'].lt(.05).any()
assert not frames['gee']['p_fdr_gee_exploratory'].lt(.05).any()
DELTA=float(np.log(21/11))
queries={
 'industry': "SELECT i.*, e.nobs_industry AS model_rows, e.area_clusters_industry AS model_areas FROM industry i JOIN effects e USING(industry_id) WHERE e.scenario='main_500' AND e.outcome='log_sales' ORDER BY i.industry_id",
 'district': 'SELECT * FROM district ORDER BY district',
 'store_size': "SELECT *, CASE total_store_size_group WHEN '0' THEN 0 WHEN '1~2' THEN 1 WHEN '3~5' THEN 2 WHEN '6~10' THEN 3 WHEN '11~50' THEN 4 WHEN '51+' THEN 5 ELSE 6 END AS display_order FROM store_size ORDER BY display_order",
 'sales':'SELECT * FROM sales ORDER BY industry_id',
 'night':'SELECT * FROM night ORDER BY industry_id',
}


In [ ]:
queries['sensitivity'] = """
WITH p AS (SELECT scenario,
 MAX(CASE WHEN outcome='log_sales' THEN p_value END) AS sales_p,
 MAX(CASE WHEN outcome='night_share_pct' THEN p_value END) AS night_p,
 MAX(CASE WHEN outcome='log_sales' THEN p_holm_primary END) AS sales_holm,
 MAX(CASE WHEN outcome='night_share_pct' THEN p_holm_primary END) AS night_holm
 FROM omnibus GROUP BY scenario),
 m AS (SELECT scenario, MAX(label) AS label, MAX(nobs_total) AS model_rows,
 MAX(area_clusters_total) AS model_areas FROM samples GROUP BY scenario)
SELECT m.*,p.sales_p,p.night_p,p.sales_holm,p.night_holm,
 CASE m.scenario WHEN 'main_500' THEN 1 WHEN 'radius_300' THEN 2
 WHEN 'radius_800' THEN 3 WHEN 'balanced_500' THEN 4 WHEN 'stores_500' THEN 5 ELSE 6 END AS display_order
 FROM m JOIN p USING(scenario) ORDER BY display_order"""
queries['signals'] = f"""
SELECT e.scenario,s.label,e.industry_id,e.industry,e.p_value,
 e.coefficient*{DELTA} AS estimate,e.ci_low*{DELTA} AS ci_low,e.ci_high*{DELTA} AS ci_high,
 CASE e.scenario WHEN 'main_500' THEN 1 WHEN 'radius_300' THEN 2
 WHEN 'radius_800' THEN 3 WHEN 'balanced_500' THEN 4 WHEN 'stores_500' THEN 5 ELSE 6 END AS display_order
 FROM effects e JOIN samples s ON e.scenario=s.scenario AND e.outcome=s.outcome
 WHERE e.outcome='night_share_pct' AND e.p_value<0.05 ORDER BY display_order,e.industry_id"""
queries['comparison'] = """
SELECT c.industry_id,c.industry,
 MAX(CASE WHEN c.outcome='log_sales' THEN c.coefficient_fe END) AS sales_fe,
 MAX(CASE WHEN c.outcome='log_sales' THEN c.coefficient_gee END) AS sales_gee,
 MAX(CASE WHEN c.outcome='night_share_pct' THEN c.coefficient_fe END) AS night_fe,
 MAX(CASE WHEN c.outcome='night_share_pct' THEN c.coefficient_gee END) AS night_gee,
 MAX(CASE WHEN g.outcome='night_share_pct' THEN g.p_fdr_gee_exploratory END) AS gee_night_bh
 FROM comparison c JOIN gee g USING(industry_id,outcome) GROUP BY c.industry_id,c.industry ORDER BY c.industry_id"""
queries['information'] = """SELECT i.*,s.areas_spanning_10_to_20,
 100*i.top5_information_share AS top5_pct
 FROM information i JOIN support s USING(industry_id,outcome)
 WHERE i.outcome='log_sales' ORDER BY i.industry_id"""
rows={key:pd.read_sql_query(query,connection) for key,query in queries.items()}
connection.close()
assert len(rows['signals'])==9
assert rows['sensitivity'][['sales_p','night_p']].gt(.05).all().all()
print('검토 표:',{key:len(frame) for key,frame in rows.items()})


## 2. 정확한 단위·출처와 표 표현

회귀표의 p값은 반올림으로 유의성 경계가 달라 보이지 않도록 소수 넷째 자리까지 표시한다. 신뢰구간은 개별 95% 구간이며 BH 보정은 p값에만 적용된다. 원 수치는 보고서 데이터에 함께 남긴다.

In [ ]:
for key in ['sales','night','signals']:
    f=rows[key]
    f['estimate_text']=f['estimate'].map(lambda x:f'{x:+.3f}')
    f['interval_text']=[f'[{a:+.3f}, {b:+.3f}]' for a,b in zip(f.ci_low,f.ci_high)]
    f['p_text']=f['p_value'].map(lambda x:f'{x:.4f}')
    if 'p_fdr_primary' in f: f['bh_text']=f['p_fdr_primary'].map(lambda x:f'{x:.4f}')
for col in ['sales_p','night_p','sales_holm','night_holm']:
    rows['sensitivity'][col+'_text']=rows['sensitivity'][col].map(lambda x:'해당 없음' if pd.isna(x) else f'{x:.4f}')
for col in ['sales_fe','sales_gee','night_fe','night_gee','gee_night_bh']:
    rows['comparison'][col+'_text']=rows['comparison'][col].map(lambda x:f'{x:.3f}')
rows['industry']['median_sales_억원']=rows['industry']['median_observed_quarter_sales_won']/1e8
for key,frame in rows.items(): frame.to_csv(OUT/(key+'_report_table.csv'),index=False,encoding='utf-8-sig')
source_tables={'industry':['industry','effects'],'district':['district'],'store_size':['store_size'],
 'sales':['sales'],'night':['night'],'sensitivity':['omnibus','samples'],
 'signals':['effects','samples'],'comparison':['comparison','gee'],'information':['information','support']}
sources=[]
for key,query in queries.items():
    names=source_tables[key]
    sources.append({'id':key,'label':key+' 검증된 보고서 입력','path':paths[names[0]].relative_to(ROOT).as_posix(),
        'query':{'engine':'sqlite','language':'sql','sql':query,
        'description':'저장된 최종 분석 CSV를 동명 임시 테이블로 적재해 보고서 행을 생성. 회귀는 최종 05·06 노트북에서 계산.',
        'tables_used':names,'files_used':[paths[name].relative_to(ROOT).as_posix() for name in names],
        'transformation_context':'표 표시용 소수점 및 개별 신뢰구간 문자열 서식; 원 수치 유지'}})
for key,path in [('execution',R/'execution_manifest.json'),('scope',R/'08_food_scope_metadata.json'),
 ('rdf',R/'06_knowledge_graph_summary.json'),('tools',R/'07_tool_readiness_summary.json'),
 ('fe_figure',R/'figures/09_외식업10종_기본결과.png'),('gee_figure',R/'figures/10_fe_gee_coefficients.png')]:
    sources.append({'id':key,'label':path.name,'path':path.relative_to(ROOT).as_posix()})


## 3. 본문과 근거 배치

결론 → 지표·표본 → 주요 추정 결과 → 민감도·보조모형 → 구현 성과 → 한계와 후속 질문의 순서다. 각 그림·표 바로 앞뒤에 읽는 법과 해석을 배치한다.

In [ ]:
title='공연 활동과 주변 외식업 매출의 관계: 종로구·마포구 분석 보고서'
blocks=[];tables=[];charts=[]
def md(key,body,source=None):
    b={'id':key,'type':'markdown','body':body}
    if source:b['sourceId']=source
    blocks.append(b)
def table(key,title,columns,sort='industry_id'):
    tables.append({'id':key,'title':title,'dataset':key,'sourceId':key,
       'defaultSort':{'field':sort,'direction':'asc'},
       'columns':[{'field':field,'label':label,'type':'text' if kind=='text' else 'number',
                   **({} if kind=='text' else {'format':'number'})} for field,label,kind in columns]})
    blocks.append({'id':key+'_table','type':'table','tableId':key})
def chart(key,title,x,y,kind='horizontalBar'):
    charts.append({'id':key+'_chart','title':title,'dataset':key,'sourceId':key,'type':kind,
        'subtitle':'종로구·마포구 · 2023~2025년 · 상권×업종×분기','showDescription':True,
        'valueFormat':'number','unit':'%', 'encodings':{
        'x':{'field':x,'type':'nominal','label':'업종' if key=='industry' else '전체점포 수'},
        'y':{'field':y,'type':'quantitative','label':'매출 제공률(%)'}}})
    blocks.append({'id':key+'_chart_block','type':'chart','chartId':key+'_chart'})
def picture(key,path,alt):
    encoded=base64.b64encode(path.read_bytes()).decode('ascii')
    blocks.append({'id':key+'_image','type':'html','sourceId':key,
        'body':'<figure><img style="max-width:100%;height:auto" alt="'+html.escape(alt)+'" src="data:image/png;base64,'+encoded+'" /></figure>'})
main=frames['effects'].query("scenario=='main_500'")
lookup=main.set_index(['industry_id','outcome'])
contrasts=pd.concat([rows['sales'],rows['night']]).set_index(['industry_id','outcome'])
md('title','# '+title+'\n\n**졸업 프로젝트 · 실행 결과 및 해석**  \n분석 기간: 2023~2025년 12분기 | 최종 실행: 2026년 10월 5일 03:52 KST')
md('summary','## 요약: 업종별 차이를 확정할 근거는 충분하지 않았다\n\n공연장 주변의 등록 공연 수가 늘어나는 분기에 외식업 매출과 소비 시간대가 어떻게 달라지는지 분석했다. 기본 500m 모형은 **134개 상권의 7,646개 상권·업종·분기 관측**을 사용했다. 업종별 총매출·야간 매출 비중의 20개 검정은 다중검정 보정 후 모두 5% 기준을 통과하지 못했다. 업종 간 기울기 차이도 총매출 Holm p=0.787, 야간 비중 Holm p=0.378로 뚜렷하게 구분되지 않았다.\n\n한식의 야간 비중 감소와 치킨의 야간 비중 증가가 개별 검정에서 관측됐지만, 보정 후 확정적인 결과로 채택할 수 없었다. 일부 신호는 거리·표본에 따라 달라졌다. **이 결과는 공연의 경제적 효과가 없다는 결론이 아니다.** 공개자료의 관측 범위와 추정 불확실성 안에서 업종별 관계를 구분할 근거가 부족했다는 의미다.\n\n데이터 연결·통계 계산의 재현성은 확인했다. RDF와 분석 도구를 통해 결과의 출처·단위·집계 규칙을 조회할 수 있으나, 자연어 에이전트 웹 서비스의 성능까지 검증한 연구는 아니다.')


In [ ]:
md('question','## 1. 무엇을 분석했는가\n\n공연 관람은 관람 전후의 식사·카페 이용으로 이어질 가능성이 있다. 그러나 공연이 많은 곳의 매출이 높다는 비교만으로는 관광 수요, 상권 규모, 교통 접근성과 같은 차이를 구분하기 어렵다. 본 분석은 **동일 상권·동일 업종의 분기별 변화**를 이용하고, 모든 상권에 공통으로 작용하는 업종별 분기 변화도 조정했다.\n\n연구 질문은 두 가지다. 첫째, 주변 공연 활동과 분기 총매출·야간 매출 비중 사이에 어떤 관련성이 있는가? 둘째, 그 관련성은 공개자료의 외식업 10종 사이에서 다른가? 관람객 개인의 소비나 특정 공연일의 효과를 직접 추적한 분석은 아니다.')
md('definitions','## 2. 분기 자료와 공연 ID를 같은 기준으로 연결했다\n\n분석 단위는 **상권×업종×분기**다. 한식·중식·일식·양식·제과·패스트푸드·치킨·분식·호프 및 간이주점·커피 및 음료의 10종을 포함한다.\n\n- **공연 활동:** 공연 기간이 해당 분기와 겹치고, 공연장 좌표에서 현재 상권 폴리곤까지의 최단 직선거리가 500m 이내인 고유 공연 ID 수. 같은 장기 공연은 여러 분기에 포함될 수 있다. 회차·관객 수·판매 티켓 수가 아니다.\n- **총매출:** 공개자료의 분기 추정매출 금액. 모형에서는 ln(1+총매출)을 사용한다. 물가 조정 전 명목 금액이다.\n- **야간 매출 비중:** 21~24시 매출 ÷ 총매출 × 100. 모형의 단위는 퍼센트포인트(%p)다. 18~21시 소비와 야간 매출액 자체는 다른 지표다.\n- **점포 수:** 일반점포, 프랜차이즈, 두 집단을 합한 전체점포를 구분한다. 매출 모집단과 점포 집계 모집단의 일치를 확인하지 못했으므로 점포당 매출은 분석하지 않았다.\n\n미제공 매출은 0으로 바꾸지 않았다. 거리 계산은 EPSG:5181 좌표계에서 수행했으며, 실제 보행거리나 개별 음식점까지의 거리는 아니다.','scope')
md('sample','## 3. 매출 자료는 전체 조합의 42.3%에서 관측됐다\n\n151개 상권×10업종×12분기의 **18,120행 중 매출이 제공된 행은 7,661개(42.3%)**다. 그중 한 번만 관측된 상권×업종의 15행을 제외해 회귀 표본 7,646행을 구성했다.\n\n마포구는 77개 상권에서 4,326/9,240행(46.8%), 종로구는 74개 상권에서 3,335/8,880행(37.6%)이 관측됐다. 이 차이는 매출 성장이나 상권 활력의 차이가 아니라 **자료가 관측되는 범위의 차이**다.','district')
table('district','지역별 관측 범위',[('district','지역','text'),('area_count','상권 수','number'),('sales_observed_rows','매출 제공 행','number'),('expected_rows','전체 조합','number'),('sales_coverage_pct','제공률(%)','number'),('fe_rows','회귀 관측행','number')],sort='district')
md('coverage_read','### 한식·카페와 소규모 업종의 분석 범위가 다르다\n\n아래 그림의 분모는 업종마다 동일한 1,812개 상권·분기 조합이다. 한식은 82.2%, 커피·음료는 72.2%에서 매출이 제공됐지만 패스트푸드는 24.6%, 중식은 25.6%였다. 따라서 업종별 계수 차이에는 분석 지역과 점포 규모 구성의 차이가 섞일 수 있다.','industry')
chart('industry','업종별 매출 자료 제공률','industry','sales_coverage_pct')
md('descriptive_read','아래 중앙값은 매출이 제공된 **상권·업종·분기 한 행의 총매출 중앙값**이다. 개별 점포 매출이나 해당 업종의 자치구 총매출이 아니다. 같은 업종 안에서도 큰 상권과 작은 상권이 섞이고 상권 경계가 중첩될 수 있으므로, 이 표로 업종의 수익성을 순위화하지 않는다. 가중 야간 비중은 야간·총매출이 모두 유효한 관측행에서 야간 매출 합계를 같은 행의 총매출 합계로 나눈 값이다. 상권 중첩을 제거한 자치구 전체 비중은 아니다.','industry')
table('industry','업종별 관측 표본과 매출 분포',[('industry_id','코드','text'),('industry','업종','text'),('sales_observed_rows','매출 제공 행','number'),('model_areas','회귀 상권 수','number'),('median_sales_억원','분기 총매출 중앙값(억원)','number'),('weighted_night_share_pct','관측매출 가중 야간 비중(%)','number')])


In [ ]:
md('selection','### 소규모 집단이 빠지는 선택 문제는 남아 있다\n\n전체점포가 1~2개인 **4,430행은 매출 제공률이 0%**였다. 3~5개인 집단은 66.6%, 51개 이상은 100%였다. 이 패턴은 자료 제공 여부가 점포 규모와 강하게 연관돼 있음을 보여 준다. 원천 자료의 미제공 사유 자체를 확인한 것은 아니다. 결측을 0으로 바꾸지 않은 처리는 필요하지만, 그것만으로 관측 표본이 전체 외식업을 대표하게 되지는 않는다.','store_size')
chart('store_size','전체점포 규모별 매출 자료 제공률','total_store_size_group','sales_coverage_pct','bar')
md('selection_note','그림의 ‘미제공’은 점포 수도 제공되지 않은 행이고, 0은 전체점포 수가 0으로 기록된 행이다. 두 집단을 점포 수가 양수인 집단과 구분해 읽는다. 원패널에서 10종 매출이 모두 있는 공통표본은 171개 상권·분기이며, 회귀의 최소 관측 조건을 적용하면 **16개 상권의 170개 상권·분기, 1,700행**이다. 기본모형의 약 22.2%에 불과하므로, 기본 업종 비교가 모두 동일한 상권을 비교한 결과는 아니다.')
md('model','## 4. 모형은 상권의 고정된 차이와 분기 공통 변화를 조정했다\n\n주모형은 업종별 기울기를 허용한 고정효과 패널 회귀다. a는 상권, g는 업종, t는 연도분기다.\n\n**Yₐɡₜ = αₐɡ + λɡₜ + βɡ × ln(1+공연ID수ₐₜ) + εₐɡₜ**\n\nαₐɡ는 상권×업종의 시간에 따라 변하지 않는 특성, λɡₜ는 업종×연도분기의 공통 변화를 조정한다. βɡ는 이 두 요인을 조정한 뒤 남는 공연 활동 변화와 결과변수의 관련성이다. 총매출과 야간 비중을 각각 추정했다.\n\n같은 상권에 속한 여러 업종·분기의 오차 상관을 반영하도록 상권별 군집 표준오차를 사용했다. 기본모형은 134개 군집, 자유도 133의 t/F 추론을 사용하며 소표본 보정을 포함한다. 다른 상권 사이의 공간 상관까지 처리한 것은 아니다.\n\n**두 종류의 검정을 구분한다.** 업종별 β가 0인지 보는 20개 검정에는 BH FDR 보정을 적용했다. 업종들의 β가 모두 같은지 보는 두 검정에는 Holm 보정을 적용했다. 후자는 모든 β가 0인지를 검정하는 것이 아니다. 그림·표의 95% 신뢰구간은 개별 구간이며 다중비교 동시 구간이 아니다.')
md('figure_read','## 5. 총매출에서는 양·음의 관계를 뚜렷하게 구분하지 못했다\n\n그림의 점은 업종별 기울기, 가로선은 개별 95% 신뢰구간, 점선은 0이다. 왼쪽은 ln(1+총매출), 오른쪽은 야간 비중(%p)이므로 두 축의 숫자 크기를 직접 비교하면 안 된다. 모든 점이 회색인 것은 20개 검정 중 BH 보정 후 5% 기준을 통과한 항목이 없기 때문이다. 신뢰구간이 0을 포함하는 것은 관계가 정확히 0이라는 뜻이 아니다.')
picture('fe_figure',R/'figures/09_외식업10종_기본결과.png','업종별 고정효과 기울기와 개별 95% 신뢰구간. 총매출 10종 구간은 모두 0을 포함한다.')
md('contrast_definition','### 공연 10개와 20개를 비교하면 어느 정도인가\n\n아래 표는 해석을 돕기 위한 모형상 비교다. 설명변수 차이 Δ=ln(21)−ln(11)=0.646627을 기울기에 곱했다. 총매출은 **100×[exp(β×Δ)−1]**로 역변환했다. 이는 조건부 **(총매출+1)의 기하평균 상대차이(%)**이며, 산술평균 매출 증가율이나 공연을 늘렸을 때의 인과효과가 아니다.\n\n전체 관측 범위에는 10·20이 포함되지만, 같은 상권의 최솟값≤10·최댓값≥20인 곳은 업종별 4~21개다. 정확히 10개와 20개인 분기를 관측한 전후 비교도 아니다. 비교값의 크기와 함께 신뢰구간을 읽는다.')
table('sales','공연 ID 10개→20개 모형상 비교: 총매출',[('industry_id','코드','text'),('industry','업종','text'),('estimate_text','기하평균 상대차이(%)','text'),('interval_text','개별 95% CI(%)','text'),('p_text','보정 전 p','text'),('bh_text','BH 보정 p','text'),('area_clusters_industry','상권 수','number')])
md('sales_interpretation','양식음식점의 점추정치는 +5.32%로 가장 크지만, 개별 95% 구간은 **−0.88~+11.91%**, BH 보정 p는 0.623이다. 따라서 ‘양식이 가장 큰 수혜 업종’이라는 결론을 내릴 수 없다. 일식은 −2.41%이지만 구간이 −11.02~+7.03%로 양·음의 가능성을 모두 포함한다. 한식과 카페도 각각 −0.40%, −0.84%이며 구간이 0을 포함한다.\n\n통계적으로 뚜렷한 결과가 없다는 점과 추정치가 충분히 작다는 점은 별개의 주장이다. 일부 업종의 구간 폭은 실질적으로 의미 있을 수 있는 변화를 포함하므로, ‘공연 활동은 매출과 무관하다’거나 ‘경제적 영향이 미미하다’는 결론도 현재 결과만으로는 뒷받침되지 않는다.','sales')


In [ ]:
md('night_heading','## 6. 야간 비중에는 탐색적 신호가 있지만 확정적 결과는 아니다\n\n야간 비교값은 β×Δ이며 단위는 %p다. 예를 들어 10%에서 11%가 되면 +1%p다. 비중의 변화는 야간 매출액의 변화와 같지 않다. 총매출이 더 빠르게 늘어나면 야간 매출액이 줄지 않아도 야간 비중은 낮아질 수 있다.')
table('night','공연 ID 10개→20개 모형상 비교: 21~24시 매출 비중',[('industry_id','코드','text'),('industry','업종','text'),('estimate_text','비중 차이(%p)','text'),('interval_text','개별 95% CI(%p)','text'),('p_text','보정 전 p','text'),('bh_text','BH 보정 p','text'),('area_clusters_industry','상권 수','number')])
md('night_interpretation','한식은 **−0.362%p**(개별 95% CI −0.682~−0.043), 치킨은 **+1.027%p**(0.222~1.832)였다. 두 구간은 0을 제외하고 보정 전 p도 각각 0.0265, 0.0128이다. 그러나 20개 검정의 BH 보정 p는 **0.2655와 0.2561**로 5% 기준을 통과하지 못했다.\n\n방향이 반대라는 사실은 흥미로운 후속 가설을 제시하지만, 관람 후 치킨 소비가 늘었다거나 한식 소비가 줄었다고 설명할 근거는 아니다. 관람객을 식별하지 않았고 야간 매출액의 변화도 직접 검정하지 않았다. 특히 ‘한 업종은 유의하고 다른 업종은 유의하지 않다’는 비교는 두 업종의 차이가 유의하다는 검정이 아니다.','night')
md('omnibus_heading','## 7. 업종 간 차이를 구분하지 못했다는 결론은 여러 설정에서 유지됐다\n\n아래 p값은 업종들의 기울기가 같은지 검정한 결과다. 기본모형의 Holm 보정 p는 총매출 0.787, 야간 비중 0.378이다. 반경·표본·점포 통제를 바꾼 모든 설정에서도 보정 전 동일성 검정 p가 0.05보다 컸다. **같다는 가설을 기각하지 못한 것이며, 업종들이 실제로 동일함을 입증한 것은 아니다.**\n\n균형표본은 업종별로 12분기 자료가 모두 있는 상권을 고른 것으로, 10종 모두가 같은 상권에 있어야 한다는 조건은 아니다. 일반점포·전체점포 통제 역시 매출 모집단을 확인하거나 인과효과를 식별하는 절차는 아니다.','sensitivity')
table('sensitivity','설정별 업종 기울기 동일성 검정',[('display_order','순서','number'),('label','설정','text'),('model_rows','관측행','number'),('model_areas','상권','number'),('sales_p_text','총매출 p','text'),('night_p_text','야간 p','text'),('sales_holm_text','총매출 Holm','text'),('night_holm_text','야간 Holm','text')],sort='display_order')
md('signals_read','### 개별 업종의 야간 신호는 설정에 따라 달라졌다\n\n총매출의 개별 검정은 6개 설정 모두에서 p<0.05인 항목이 없었다. 그러나 야간 비중은 아래 **9개 개별 검정**에서 보정 전 p<0.05였다. 이 표는 그 항목을 전부 보여 준다. 비주모형의 다중검정 보정값은 산출하지 않았으므로 이 결과를 확증적 발견으로 취급하지 않는다. 설정별 공연 지표의 공간 범위가 달라 같은 10→20 ID 비교도 동일한 개입을 의미하지 않는다.','signals')
table('signals','기본·민감도 분석에서 보정 전 p<0.05인 야간 항목 전체',[('display_order','설정 순서','number'),('label','설정','text'),('industry','업종','text'),('estimate_text','10→20 비교(%p)','text'),('interval_text','개별 95% CI(%p)','text'),('p_text','보정 전 p','text')],sort='display_order')
md('sensitivity_interpretation','300m에서는 양식이 −0.948%p(p=0.0203)였지만 기본 500m에서는 같은 신호가 뚜렷하지 않았다. 한식의 기본 음의 관계도 300m에서 기울기가 +0.061(p=0.854)로 바뀌었다. 치킨의 기울기는 6개 설정 모두 양수였으나 300m·800m·균형표본에서 개별 5% 유의성이 유지되지는 않았다.\n\n따라서 큰 결론인 ‘업종 간 차이를 뚜렷하게 구분하지 못함’과, 개별 업종의 방향·크기가 설정에 민감하다는 사실을 함께 보고해야 한다. 유의한 거리나 표본만 골라 주모형을 바꾸지 않았다.')


In [ ]:
md('gee_heading','## 8. GEE에서도 다중검정 보정 후 뚜렷한 within 관계는 없었다\n\nGEE(일반화추정방정식)는 상권×업종별 관측기간 평균 공연 활동과 그 평균에서의 분기 편차를 분리했다. 업종별 자치구·분기를 조정하고 Gaussian identity, 상권 내 exchangeable 작업상관과 robust 공분산을 사용했다. FE와 달리 상권×업종 고정효과는 두지 않는다. 따라서 두 모형의 차이는 상관구조 하나의 영향으로 분리할 수 없다.\n\nFE와 GEE의 관측 키는 결과변수별 7,646개로 완전히 일치했다. GEE within의 20개 검정도 별도 BH 보정 후 유의한 항목이 없었다. 다음 표는 **변환 전 기울기 β** 비교다. 총매출 β는 로그 단위, 야간 β는 %p 단위이며 모두 ln(1+공연 ID 수)가 1 증가하는 비교다. 10→20 비교값과 혼동하지 않는다.')
table('comparison','FE와 GEE의 업종별 within 기울기',[('industry_id','코드','text'),('industry','업종','text'),('sales_fe_text','매출 FE β','text'),('sales_gee_text','매출 GEE β','text'),('night_fe_text','야간 FE β','text'),('night_gee_text','야간 GEE β','text'),('gee_night_bh_text','GEE 야간 BH p','text')])
md('gee_interpretation','한식 야간 기울기는 FE −0.560에서 GEE −0.580, 치킨은 +1.589에서 +1.849로 방향이 같았다. 반면 분식 총매출은 +0.015에서 −0.023으로 부호가 달라졌고, 일식 야간 기울기도 −0.458에서 −1.089로 크기가 달라졌다. ‘모든 추정치가 안정적’이라고 요약할 수는 없다. GEE의 한식·치킨 야간 BH 보정 p는 모두 0.141이었다.\n\n두 모형의 계수 차이에 대한 별도 검정은 하지 않았다. 결과가 비슷하면 두 설정에서 비슷한 패턴을 보였다는 의미이며, GEE가 FE의 인과적 타당성을 확인하거나 어느 모형이 우월함을 입증하는 것은 아니다. between 성분은 상권 간 평균의 관계이고 별도 다중검정 보정을 하지 않아 핵심 발견으로 채택하지 않았다.','comparison')
md('information_read','## 9. 정보가 소수 상권에 집중되는 업종은 해석에 더 주의해야 한다\n\n상권 수가 많아 보여도 실제 계수 추정에 필요한 공연 활동 변화가 소수 상권에서 주로 생길 수 있다. 아래의 ‘상위 5개 정보 비중’은 고정효과를 제거한 공연 활동의 제곱합을 상권별로 나눈 기술 지표다. 중식은 53.6%, 패스트푸드는 52.3%였다. 이는 표준오차를 새로 보정한 결과나 추론상 유효 자유도가 아니다.\n\n‘10~20 범위 상권’은 관측된 최소 ID 수≤10, 최대 ID 수≥20인 상권이다. 해당 비교의 관측 지원 범위를 보여 주지만 정확히 10개와 20개인 두 분기를 의미하지 않는다.','information')
table('information','업종별 추정 정보와 비교 범위',[('industry_id','코드','text'),('industry','업종','text'),('area_clusters','명목 상권 수','number'),('areas_spanning_10_to_20','10~20 범위 상권','number'),('top5_pct','상위 5개 정보 비중(%)','number')])


In [ ]:
rdf=read_json(R/'06_knowledge_graph_summary.json')
tools=read_json(R/'07_tool_readiness_summary.json')
md('implementation','## 10. 재현 가능한 연결·조회 기반을 구축했다\n\nKOPIS의 시설 상세 1,755개와 후보 공연 상세 13,727개를 고정 입력으로 사용했다. 기본 500m 분석에 기여하는 고유 공연 12,839개는 모두 공식 시설 ID로 연결됐다. 서울 전체 목록의 나머지 이름 연결 후보와 분석에 기여하는 공연을 구분했다. 공식 ID 확인은 과거 시설 위치나 목록 밖 공연의 완전성까지 입증하지 않는다.\n\n연결 검증 55개, 외식업 데이터 검증 18개를 통과했고, 빈 최종 출력 폴더에서 분석 8개 노트북을 새 커널로 다시 실행했다. 정리 전후 기본 FE·GEE의 계수·표준오차·구간·p값은 동일했다. 이 검증은 계산과 파일 연결의 재현성을 확인하며 연구 설계의 한계를 해결하지는 않는다.')
md('rdf_scope',f'RDF에는 151개 상권, 801개 시설, 12,839개 공연과 18,120개 매출 관측 슬롯을 연결했다. 데이터 트리플은 {rdf["triples"]:,}개이며, 미제공 매출 10,459개에는 금액을 저장하지 않았다. SPARQL과 pandas 집계 대조를 포함한 검증 14개를 통과했다. 야간 비중·점포 수·회귀 결과는 현재 RDF에 포함하지 않는 부분 그래프다.','rdf')
md('tools_scope',f'분석 도구는 지표 정의·매출 집계·회귀 결과·업종 차이 검정·품질 조회의 {tools["tool_count"]}개다. 총 {tools["calls"]}회 호출과 계약 검사 {tools["contract_checks_passed"]}/{tools["contract_checks_total"]}개를 확인했다. 분기 결과를 일별·인과효과로 요청하는 등 지원 범위를 벗어나는 호출은 거절한다. 현재는 구조화된 인수를 받는 오프라인 도구이며 **LLM 자연어 계획, 자연어 파서, 웹 서비스는 연결하지 않았다.** 따라서 에이전트 성능이나 온톨로지의 오류 감소 효과를 검증했다고 주장하지 않는다.','tools')
md('limits','## 11. 이 보고서가 말할 수 없는 것\n\n**인과효과:** 공연과 함께 변하는 관광·지역 행사·경기·점포 구성 등의 요인을 모두 조정하지 않았다. 매출이 높은 상권에 공연이 늘어나는 역방향 관계도 배제하지 못한다. 고정효과만으로 이 문제들이 사라지지는 않는다.\n\n**관람 전후 소비:** 분기 매출과 공연 ID 수는 개별 공연일·관객을 식별하지 않는다. 21~24시 비중의 변화만으로 관람 전후 식사·카페 이용을 확인할 수 없다. 하루짜리와 장기 공연을 ID 하나로 세므로 실제 공연 강도에도 오차가 남는다.\n\n**전체 외식업 대표성:** 소규모 집단에 미제공이 집중되고 업종별 관측 지역이 다르다. 상권 경계 중첩을 제거한 자치구 매출 합계도 아니다. 매출액으로 점포당 수익성이나 업종 규모를 비교하는 연구로 확대하면 안 된다.\n\n**공간·역사적 동일성:** 하나의 공연이 인접한 여러 상권에 포함될 수 있다. 현재 군집 표준오차는 서로 다른 상권의 잔차 상관을 처리하지 않는다. 현재 경계·시설 좌표를 2023~2025년에도 같은 것으로 적용한 한계도 남는다.\n\n**확인적 검증:** 두 업종의 선행 결과를 본 뒤 10종으로 확장한 탐색 분석이다. BH·Holm 보정은 여러 검정을 함께 읽기 위한 절차이며, 모든 연구 선택에 따른 불확실성을 없애지는 않는다.')
md('conclusion','## 12. 논문에서 채택할 결론과 후속 질문\n\n현재 자료에서는 공연 활동과 외식업 매출의 관계가 업종별로 다르다는 근거를 뚜렷하게 확인하지 못했다. 한식·치킨의 야간 비중에서 탐색적 신호가 있었으나 다중검정과 설정 변경을 고려하면 확정적인 업종별 반응으로 해석하기 어렵다. 이를 근거로 특정 업종의 수혜를 예측하거나 공연 유치의 매출 효과를 단정해서는 안 된다.\n\n이 연구의 확인 가능한 성과는 서로 다른 공개자료를 공식 ID·공간·분기 기준으로 연결하고, 업종별 관계와 불확실성을 같은 절차로 비교하며, 자료 제공 범위가 해석을 얼마나 제한하는지 드러냈다는 데 있다. 비유의 결과를 숨기거나 유의한 설정만 선택하지 않고 범위와 한계를 함께 제시하는 것이 현재 졸업논문의 적절한 결론이다.\n\n후속 연구에서는 다음 질문을 우선 검토할 수 있다. **첫째**, 상연 횟수·관객 수 또는 정확한 공연일을 확보했을 때 같은 패턴이 나타나는가? **둘째**, 매출 미제공 기준과 과거 상권 경계·시설 이력을 확인하면 비교 가능한 표본이 달라지는가? **셋째**, 상권 사이의 잔차 공간 상관을 점검하고 더 긴 기간이나 새로운 자료로 분석하면 신호가 재현되는가? 이는 추가 자료와 설계가 필요한 질문이며 이번 결과가 이미 답한 내용은 아니다.')
md('sources_note','## 자료와 재현 안내\n\n원천 자료는 서울시 상권분석서비스의 추정매출·점포·상권영역 자료와 KOPIS 공연·공연장 자료다. 이 보고서는 최종 실행의 고정 결과를 해석한 스냅샷이며 새 데이터 수집이나 모형 재적합을 하지 않았다. 각 표·그림의 출처 정보에서 실제 입력 파일과 보고서용 쿼리를 확인할 수 있다.\n\n주분석: 최종 노트북 05, GEE: 06, 데이터·연결: 01~04, RDF·도구: 07~08. 보고서의 기술통계와 수치 대조는 별도 보고서 근거 노트북에 기록했다. 수치·입력 파일 해시와 HTML 구조를 검증했다. 자동 브라우저 실행 검증은 환경 제약으로 수행하지 못했다.')


## 4. 단일 HTML 보고서 저장과 검증

과학적 신뢰구간 그림은 기존 Matplotlib 산출물을 그대로 포함한다. 범주별 제공률은 보고서의 기본 차트로 작성한다. 이미지·데이터를 HTML에 내장하여 다른 파일이나 서버 없이 읽을 수 있도록 한다.

In [ ]:
generated=datetime.now(timezone(timedelta(hours=9))).isoformat()
artifact={'surface':'report','manifest':{'version':1,'surface':'report','title':title,
 'generatedAt':generated,'description':'최종 실행 결과의 통계적 해석과 한계를 담은 졸업 프로젝트 분석 보고서',
 'blocks':blocks,'tables':tables,'charts':charts,'sources':sources},
 'snapshot':{'version':1,'generatedAt':generated,'status':'ready',
 'datasets':{key:project_json_loads(frame.to_json(orient='records',force_ascii=False)) for key,frame in rows.items()},'accessIssues':[]},
 'sources':sources}
(OUT/'artifact.json').write_text(json.dumps(artifact,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
assert all(sha(project_path(path))==digest for path,digest in input_hashes.items())
notes={'audience':'technical','delivery':'portable_html','scope':'종로구·마포구/외식업10종/2023~2025분기',
 'question':'공연 활동과 주변 외식업 매출의 관련성과 업종별 차이',
 'source_sha256':input_hashes,'models_refit':False,
 'required_structure_mapping':{'title':'title','technical_summary':'summary','key_findings':'figure_read/night_heading/omnibus_heading',
 'definitions':'definitions/sample','methodology':'model/gee_heading','limitations':'selection/information_read/limits',
 'next_steps_and_questions':'conclusion'},
 'structure_note':'지표 정의와 표본을 결과보다 먼저 제시해 분모·단위 오독 방지',
 'chart_map':[{'id':'industry_chart','question':'업종별 관측 범위','type':'horizontalBar','rows':10},
 {'id':'store_size_chart','question':'점포 규모별 선택적 미제공','type':'bar','rows':7},
 {'id':'fe_figure','question':'추정치와 불확실성','type':'existing_matplotlib_dot_interval',
 'rationale':'연구용 신뢰구간 그림을 이미 검증한 최종 실행 산출물에서 재사용; 좌우축 단위 분리'}],
 'table_rationale':'정확한 개별 CI·보정 전 p·다중보정 p·표본수의 동시 조회',
 'assessment':'Share with caveats', 'missingness_zero_filled':False,'causal_claim':False}
(OUT/'report_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
print('보고서:',len(blocks),'본문·근거 블록 /',len(tables),'표 /',len(charts),'차트')


In [ ]:
builder=PROJECT_ROOT / 'environment/report-builder/skills/build-report/scripts/deliver_portable_artifact.mjs'
result=subprocess.run([NODE_EXECUTABLE,str(builder),'--input',str(OUT/'artifact.json'),
 '--output',str(OUT/'report.html')],capture_output=True,text=True,encoding='utf-8',timeout=120)
print(result.stdout[-6000:])
if result.returncode: print(result.stderr[-6000:])
assert result.returncode==0,'보고서 패키징 실패'
(OUT/'delivery_receipt.json').write_text(result.stdout,encoding='utf-8')


해석: 보고서는 최종 분석의 관측자료 관련성을 전달한다. 계산 검증·파일 검증을 인과성이나 모집단 대표성의 검증으로 확대하지 않는다. HTML 보고서가 주 산출물이며, 근거 CSV와 이 노트북은 재현 자료다.